In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier

In [2]:
np.random.seed(42)

damage_classes = [
    "Front Breakage",
    "Front Crushed",
    "Rear Breakage",
    "Rear Crushed"
]

severity_levels = [
    "Minor",
    "Moderate",
    "Severe",
    "Total Loss"
]

base_cost = {
    "Front Breakage": 40000,
    "Front Crushed": 90000,
    "Rear Breakage": 35000,
    "Rear Crushed": 80000
}

n_samples = 5000
rows = []

for _ in range(n_samples):
    damage_class = np.random.choice(damage_classes)
    vehicle_age = np.random.randint(0, 15)
    mileage = np.random.randint(0, 200000)

    repair_cost = (
        base_cost[damage_class]
        + vehicle_age * 1200
        + mileage * 0.05
        + np.random.normal(0, 5000)
    )
    repair_cost = max(1000, repair_cost)

    # Claim severity
    if repair_cost < 25000:
        claim_severity = "Minor"
    elif repair_cost < 60000:
        claim_severity = "Moderate"
    elif repair_cost < 150000:
        claim_severity = "Severe"
    else:
        claim_severity = "Total Loss"

    # Additional fraud-related features
    prior_claims = np.random.poisson(1)
    policy_age_months = np.random.randint(1, 120)
    days_since_policy_inception = np.random.randint(1, 3650)

    # Claimed amount may be inflated
    inflation_factor = np.random.uniform(0.9, 1.4)
    claim_amount = repair_cost * inflation_factor

    # Fraud score heuristic
    fraud_score = 0

    if claim_amount > repair_cost * 1.2:
        fraud_score += 2

    if prior_claims >= 3:
        fraud_score += 2

    if policy_age_months < 6:
        fraud_score += 1

    if damage_class in ["Front Crushed", "Rear Crushed"]:
        fraud_score += 1

    if claim_severity == "Total Loss":
        fraud_score += 1

    # Convert to fraud label with randomness
    probability = min(0.95, fraud_score / 8)
    fraud_flag = np.random.rand() < probability

    rows.append([
        damage_class,
        vehicle_age,
        mileage,
        round(repair_cost, 2),
        round(claim_amount, 2),
        claim_severity,
        prior_claims,
        policy_age_months,
        days_since_policy_inception,
        int(fraud_flag)
    ])

df = pd.DataFrame(
    rows,
    columns=[
        "damage_class",
        "vehicle_age",
        "mileage",
        "repair_cost",
        "claim_amount",
        "claim_severity",
        "prior_claims",
        "policy_age_months",
        "days_since_policy_inception",
        "fraud_flag"
    ]
)

df.head()

,damage_class,vehicle_age,mileage,repair_cost,claim_amount,claim_severity,prior_claims,policy_age_months,days_since_policy_inception,fraud_flag
0,Rear Breakage,3,131932,47915.32,54126.31,Moderate,1,87,331,0
1,Rear Crushed,7,41090,87377.35,115007.93,Severe,0,2,2392,1
2,Rear Crushed,11,184779,102550.06,104230.41,Severe,1,60,976,1
3,Front Crushed,2,67435,93632.79,112712.57,Severe,1,3,3557,1
4,Rear Breakage,1,141699,43760.58,44658.23,Moderate,1,2,1364,0


In [3]:
os.makedirs("../data/synthetic", exist_ok=True)

df.to_csv(
    "../data/synthetic/fraud_risk_data.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [4]:
print(df["fraud_flag"].value_counts())
print(df["fraud_flag"].value_counts(normalize=True))

fraud_flag
0    4016
1     984
Name: count, dtype: int64
fraud_flag
0    0.8032
1    0.1968
Name: proportion, dtype: float64


In [5]:
X = df.drop(columns=["fraud_flag"])
y = df["fraud_flag"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
categorical_features = [
    "damage_class",
    "claim_severity"
]

numeric_features = [
    "vehicle_age",
    "mileage",
    "repair_cost",
    "claim_amount",
    "prior_claims",
    "policy_age_months",
    "days_since_policy_inception"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

In [8]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

In [9]:
model.fit(X_train, y_train)

print("Fraud model training completed.")

Fraud model training completed.


In [10]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [11]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC : {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7820
ROC-AUC : 0.7288

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.94      0.87       803
           1       0.36      0.14      0.20       197

    accuracy                           0.78      1000
   macro avg       0.59      0.54      0.54      1000
weighted avg       0.73      0.78      0.74      1000


Confusion Matrix:
[[755  48]
 [170  27]]


In [12]:
os.makedirs("../saved_models", exist_ok=True)

joblib.dump(
    model,
    "../saved_models/fraud_model.pkl"
)

print("Fraud model saved successfully.")

Fraud model saved successfully.


In [13]:
sample = pd.DataFrame({
    "damage_class": ["Front Crushed"],
    "vehicle_age": [4],
    "mileage": [60000],
    "repair_cost": [100000],
    "claim_amount": [145000],
    "claim_severity": ["Severe"],
    "prior_claims": [4],
    "policy_age_months": [2],
    "days_since_policy_inception": [60]
})

fraud_probability = model.predict_proba(sample)[0, 1]
fraud_prediction = model.predict(sample)[0]

print(f"Fraud Probability: {fraud_probability:.2%}")
print("Fraud Flag:", "Fraud" if fraud_prediction == 1 else "Legitimate")

Fraud Probability: 42.12%
Fraud Flag: Legitimate


In [14]:
loaded_model = joblib.load(
    "../saved_models/fraud_model.pkl"
)

loaded_probability = loaded_model.predict_proba(sample)[0, 1]

print(f"Loaded Model Fraud Probability: {loaded_probability:.2%}")

Loaded Model Fraud Probability: 42.12%
